<img width="8%" alt="Archivist" src="" style="border-radius: 15%">

# Archivist - Indexing
<a href="">Give Feedback</a> | <a href="">Bug report</a>

**Tags:** #archivist #scan #embed #index #qdrant #ollama #rag #python

**Author:** [Mohamed BASRI](https://github.com/mbasri)

**Last update:** 2026-04-02 (Created: 2026-04-02)

**Description:** Full indexing pipeline — scans a directory, skips already indexed files (hash-based), chunks text content, embeds each chunk locally with Ollama, and stores vectors into Qdrant.

**References:**
- [Qdrant Python client](https://python-client.qdrant.tech/)
- [Ollama Python library](https://github.com/ollama/ollama-python)
- [nomic-embed-text](https://ollama.com/library/nomic-embed-text)
- [Archivist project](https://github.com/mbasri/archivist)

## Setup

### Install dependencies

In [ ]:
import sys
import os
import subprocess
from pathlib import Path

project_root = Path().resolve()
project_root = project_root.parent if project_root.name == "notebooks" else project_root
venv_path    = project_root / ".venv"
venv_python  = venv_path / ("Scripts/python" if os.name == "nt" else "bin/python")

if Path(sys.executable).resolve() == venv_python.resolve():
    print("✓ Running inside the Archivist venv")
else:
    if not venv_path.exists():
        print("Creating venv...")
        subprocess.run([sys.executable, "-m", "venv", str(venv_path)], check=True)

    print("Installing dependencies...")
    subprocess.run(
        [str(venv_python), "-m", "pip", "install",
         "-r", str(project_root / "requirements.txt"),
         "ipykernel", "--quiet"],
        check=True
    )
    subprocess.run(
        [str(venv_python), "-m", "ipykernel", "install",
         "--user", "--name=archivist", "--display-name=Archivist"],
        check=True
    )
    print()
    print("✓ Venv ready.")
    print("→ Select the 'Archivist' kernel: Kernel > Change kernel > Archivist")
    print("→ Then run the notebook again")

## Input

### Import libraries

In [ ]:
import hashlib
import json
import uuid
from collections import defaultdict

import ollama
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

### Setup Variables
- `nas_path`: path to the directory to scan and index
- `text_extensions`: file extensions readable as plain text (no conversion needed)
- `embedding_model`: Ollama model used to embed chunks
- `embedding_dim`: vector dimension of the model (`nomic-embed-text` → 768)
- `chunk_size`: number of characters per chunk
- `chunk_overlap`: overlapping characters between consecutive chunks
- `collection_name`: Qdrant collection name
- `qdrant_host` / `qdrant_port`: Qdrant connection
- `ollama_host`: Ollama connection

In [ ]:
nas_path        = ""
text_extensions = [".txt", ".md", ".csv"]

embedding_model = "nomic-embed-text"
embedding_dim   = 768
chunk_size      = 1000
chunk_overlap   = 100

collection_name = "archivist"
qdrant_host     = "localhost"
qdrant_port     = 6333
ollama_host     = "http://localhost:11434"

index_path = project_root / "index.json"
print(f"Index file: {index_path}")

## Model

### Load index

In [ ]:
if index_path.exists():
    with open(index_path, "r") as f:
        index = json.load(f)
else:
    index = {}

print(f"Index loaded: {len(index)} file(s) already indexed")

### Scan and filter new files

In [ ]:
def hash_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

all_files = defaultdict(int)
to_index  = []

for file in Path(nas_path).rglob("*"):
    if not file.is_file():
        continue
    ext = file.suffix.lower()
    all_files[ext] += 1
    if ext not in text_extensions:
        continue
    h = hash_file(file)
    if h not in index:
        to_index.append((file, h))

print(f"Total files scanned : {sum(all_files.values())}")
print(f"Already indexed     : {len(index)}")
print(f"To index            : {len(to_index)}")

### Initialize Qdrant

In [ ]:
qdrant = QdrantClient(host=qdrant_host, port=qdrant_port)

existing = [c.name for c in qdrant.get_collections().collections]
if collection_name not in existing:
    qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )
    print(f"Collection '{collection_name}' created")
else:
    print(f"Collection '{collection_name}' already exists")

### Chunk, embed, and index

In [ ]:
def chunk_text(text: str, size: int, overlap: int) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

ollama_client = ollama.Client(host=ollama_host)
indexed, errors = 0, []

for file, h in to_index:
    try:
        text   = file.read_text(encoding="utf-8", errors="ignore")
        chunks = chunk_text(text, chunk_size, chunk_overlap)

        points = []
        for i, chunk in enumerate(chunks):
            vector = ollama_client.embed(model=embedding_model, input=chunk)["embeddings"][0]
            points.append(PointStruct(
                id      = str(uuid.uuid4()),
                vector  = vector,
                payload = {
                    "file":         str(file),
                    "file_name":    file.name,
                    "hash":         h,
                    "chunk_index":  i,
                    "total_chunks": len(chunks),
                    "text":         chunk,
                }
            ))

        qdrant.upsert(collection_name=collection_name, points=points)

        index[h] = str(file)
        with open(index_path, "w") as f:
            json.dump(index, f, indent=2)

        print(f"  [OK]  {file.name} → {len(chunks)} chunk(s)")
        indexed += 1

    except Exception as e:
        print(f"  [ERR] {file.name} → {e}")
        errors.append((file, str(e)))

## Output

### Display result

In [ ]:
info = qdrant.get_collection(collection_name)

print(f"Files indexed           : {indexed}")
print(f"Errors                  : {len(errors)}")
print(f"Total vectors in Qdrant : {info.points_count}")

if errors:
    print("\nFailed files:")
    for file, err in errors:
        print(f"  - {file.name}: {err}")